# Assignment 3: Introduction to Spark Structured Streaming

## Name: M Vimal Krishna Rao
## Reg. No: 25MML1003
## Subject: Distributed Data Processing

## (a): Theory – Structured Streaming in Spark

## Introduction
Structured Streaming is a scalable, high-throughput, and fault-tolerant stream processing engine built on Apache Spark. It enables real-time processing of continuous data streams using the same APIs as batch processing, making development simpler and more consistent.

## What is Structured Streaming?
Structured Streaming treats incoming data as an unbounded table. Data continuously arrives, and Spark incrementally processes this data by executing queries on the table. The results are updated in real time as new data is added.

## Difference Between Structured Streaming and Batch Processing

| Feature | Structured Streaming | Batch Processing |
|--------|---------------------|------------------|
| Nature of Data | Continuous (unbounded) | Finite (bounded) |
| Processing Type | Real-time / near real-time | Periodic |
| Latency | Low latency | High latency |
| Execution | Continuous queries | One-time execution |
| Output | Incremental updates | Complete result after processing |
| Use Case | Live analytics | Historical analysis |

## Key Features of Structured Streaming

### Unified API
- Uses the same DataFrame and Dataset APIs as batch processing  
- Reduces learning curve and development effort  

### Scalability
- Handles large-scale streaming data across clusters  
- Automatically distributes workload  

### Fault Tolerance
- Uses checkpointing to recover from failures  
- Ensures no data loss during processing  

### Exactly-Once Processing
- Guarantees each record is processed only once  
- Prevents duplication and inconsistency  

### Event-Time Processing
- Processes data based on event timestamps  
- Handles late-arriving data effectively  

### Window Operations
- Supports time-based aggregations (tumbling, sliding windows)  
- Useful for real-time analytics  

### Integration with Multiple Sources
- Supports Kafka, file systems, sockets, and cloud storage  
- Enables flexible data ingestion  

### Continuous and Micro-Batch Processing
- Processes data in small batches (micro-batching) or continuous mode  
- Balances latency and performance  

## Real-Time Use Cases

### Financial Fraud Detection
- Detect suspicious transactions instantly  
- Trigger alerts in real time  

### Log and Event Monitoring
- Monitor server logs continuously  
- Identify errors and anomalies  

### IoT and Sensor Data Processing
- Analyze data from connected devices  
- Enable smart systems and automation  

### Real-Time Dashboards
- Provide live business insights  
- Track KPIs and metrics dynamically  

### Recommendation Systems
- Update recommendations based on user activity  
- Improve personalization  

### Social Media Analytics
- Analyze trends, sentiments, and user engagement in real time  

## Advantages of Structured Streaming
- Low latency processing  
- High scalability and performance  
- Easy integration with existing Spark applications  
- Simplified development using unified APIs  
- Strong fault tolerance and reliability  

## Conclusion
Structured Streaming is a powerful extension of Apache Spark that enables efficient real-time data processing. By treating streams as tables and using a unified API, it simplifies development while providing high performance, fault tolerance, and scalability. It is widely used in modern applications requiring instant insights and continuous data processing.

## (b): Write a PySpark Structured Streaming program to:
- Read data from a streaming source (e.g., socket or file directory)
- Perform a simple transformation (such as word count)
- Output the results to the console
- Explain each step in comments

# Experiment: PySpark Structured Streaming using File Source

## Aim
- To implement Structured Streaming using PySpark  
- To read streaming data from a file directory  
- To perform word count transformation  
- To display results on the console  

File-based streaming in Apache Spark simulates real-time data processing by continuously monitoring a directory for new files and treating them as incoming data streams.

In [1]:
# Install PySpark (only needed in Colab)
!pip install pyspark

## Step-by-Step Flow


In [7]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("FileStreamingWordCount") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

## 1. Creating a Streaming Source (Directory)
* A folder (e.g., /content/stream_data) is created
* This folder acts as a data source (stream)
* Spark will continuously monitor this directory

In [8]:
import os

# Create a directory for streaming input
input_dir = "/content/stream_data"
os.makedirs(input_dir, exist_ok=True)

print("Streaming directory created:", input_dir)

Streaming directory created: /content/stream_data


## 2. Initial File as First Data Batch
* When you add data1.txt, Spark reads it as the first micro-batch
* The file is treated as new incoming data
* Data is loaded into a streaming DataFrame (lines)

In [14]:
with open("/content/stream_data/data1.txt", "w") as f:
    f.write("hello spark hello world\nspark is fast")

print("Initial file added")

Initial file added


## 3. Streaming DataFrame Creation
* lines = spark.readStream.format("text").load("path")
* Creates a streaming DataFrame
* It does NOT load all data at once
* Instead, it waits for new files continuously

In [15]:
# Read streaming data from the directory
# Each new file added will be treated as new data

lines = spark.readStream \
    .format("text") \
    .load("/content/stream_data")

# 'lines' contains a column named "value"

## 4. Transformation (Word Count)
* Text is split into words using split() and explode()
* Words are grouped and counted
* These are lazy transformations (not executed yet)

In [16]:
from pyspark.sql.functions import explode, split

# Split lines into words
words = lines.select(
    explode(
        split(lines.value, " ")
    ).alias("word")
)

# Perform word count
word_counts = words.groupBy("word").count()

## 5. Starting the Stream (Trigger Execution)
* query = word_counts.writeStream.start()
* This is the action
* Spark starts processing data in micro-batches
* Execution begins here

In [17]:
# Start streaming query
query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()

print("Streaming started...")

Streaming started...


## 6. Micro-Batch Processing
* Spark processes data in small batches instead of one-by-one
* Each new file added = new micro-batch
*  Example:
* data1.txt → Batch 1
* data2.txt → Batch 2

## 7. Adding New Files (Simulating Stream)
When you add data2.txt:
* Spark detects it automatically
* Processes only the new file (not old ones)
* Updates the result

## 8. Output Updated Continuously
* Console output keeps updating
* In "complete" mode:
Entire word count table is refreshed every batch

In [18]:
import time

# Wait before adding new data
time.sleep(5)

with open("/content/stream_data/data2.txt", "w") as f:
    f.write("hello world again\nspark streaming example")

print("New file added!")

New file added!


In [20]:
# Let it run for some time, then stop
import time
time.sleep(10)

query.stop()
print("Streaming stopped")

Streaming stopped


## Result
- Successfully implemented file-based streaming in PySpark  
- Word count was computed on streaming data  
- Output was displayed continuously and updated with new files  